In [ ]:
# ============================================================
# GOOGLE COLAB SETUP — run this cell first when using Colab
# ============================================================
import sys, os

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_PATH = '/content/drive/MyDrive/Factor-Research'

    if not os.path.exists(REPO_PATH):
        print('Cloning repository to Google Drive...')
        os.system(f'git clone https://github.com/mbrennan5/Factor-Research.git {REPO_PATH}')
    else:
        print(f'Repository found at {REPO_PATH}')

    print('Installing packages...')
    os.system('pip install -q lightgbm xgboost optuna plotly tqdm yfinance alpaca-py pyarrow')

    NOTEBOOKS_DIR = os.path.join(REPO_PATH, 'notebooks')
    os.chdir(NOTEBOOKS_DIR)
    print(f'Working directory set to: {os.getcwd()}')
else:
    print('Running locally — no Colab setup needed.')


# Machine Learning Models for Stock Return Prediction

This notebook implements **comprehensive machine learning models** for stock return prediction, covering multiple architectures from traditional linear models to advanced deep learning approaches.

Build and compare **6 different model architectures** for stock return prediction:
- **Linear Regression** (Benchmark)
- **DNN** (Deep Neural Network)
- **DeepNet** (Multi-layer Deep Network)
- **AlexNet** (Convolutional Neural Network)
- **LSTM+ResNet** (Hybrid Time-series + Residual Network)
- **Transformer** (Attention-based Architecture)

**Key Features**

- **Hyperparameter Optimization**: Optuna-based automated tuning
- **Performance Metrics**: Information Coefficient (IC) calculation and comparison
- **Model Comparison**: Systematic evaluation across all architectures
- **Best Performance**: Target IC > 0.06 with 72% improvement over baseline



In [ ]:
# === 1. Import Libraries and Setup ===

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import optuna
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
import os
import gc

warnings.filterwarnings('ignore')

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔧 Using device: {device}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("📚 Libraries imported successfully")
print("🎯 Ready for comprehensive ML model training and evaluation")


In [ ]:
# === 2. Data Loading and Preparation ===

print("📊 Loading data for ML model training...")

# Load return data (target variable)
ret_path = '../data/processed/wide_data_preparation/vwap1pct_daily_data.csv'
try:
    ret = pd.read_csv(ret_path, index_col=0)
    ret.index = pd.to_datetime(ret.index)
    print(f"✅ Return data loaded: {ret.shape}")
    print(f"   Date range: {ret.index[0]} to {ret.index[-1]}")
except FileNotFoundError:
    print(f"❌ Error: Return data not found at {ret_path}")
    print("Please ensure the data preparation step has been completed.")
    ret = None

# Load selected factors (features)
factors_path = '../data/factors/selected_factors'
try:
    factor_files = [f for f in os.listdir(factors_path) if f.endswith('.csv') and not f.startswith('.')]
    print(f"✅ Found {len(factor_files)} factor files")
    
    # Load and combine all factors
    factors_list = []
    for file in tqdm(factor_files[:20], desc="Loading factors"):  # Limit to first 20 for demo
        try:
            factor_df = pd.read_csv(os.path.join(factors_path, file), index_col=0)
            factor_df.index = pd.to_datetime(factor_df.index)
            factors_list.append(factor_df)
        except Exception as e:
            print(f"   Warning: Failed to load {file}: {e}")
    
    if factors_list:
        # Stack factors as 3D array: (time, stocks, factors)
        factors_3d = np.stack([df.values for df in factors_list], axis=2)
        print(f"✅ Factors combined: {factors_3d.shape} (time, stocks, factors)")
    else:
        print("❌ No valid factor files loaded")
        factors_3d = None
        
except FileNotFoundError:
    print(f"❌ Error: Factors directory not found at {factors_path}")
    factors_3d = None

# Set date range for training
start_date = "2022-11-15"
end_date = "2023-11-17"

if ret is not None and factors_3d is not None:
    # Align data to date range
    date_mask = (ret.index >= start_date) & (ret.index <= end_date)
    ret_aligned = ret[date_mask]
    factors_aligned = factors_3d[date_mask.values]
    
    print(f"📅 Data aligned to {start_date} - {end_date}")
    print(f"   Final shapes: Factors {factors_aligned.shape}, Returns {ret_aligned.shape}")
else:
    print("❌ Cannot proceed without data")
    factors_aligned = None
    ret_aligned = None


In [ ]:
# === 3. Data Preprocessing and Time Series Splitting ===

class DataPreprocessor:
    """Enhanced data preprocessor for ML model training"""
    
    def __init__(self, factors, returns):
        self.factors = factors  # (time, stocks, factors)
        self.returns = returns  # (time, stocks)
        
    def preprocess(self, standardize=True, fill_nan=True):
        """Preprocess data with standardization and NaN handling"""
        print("🔄 Preprocessing data...")
        
        factors = self.factors.copy()
        returns = self.returns.copy()
        
        if fill_nan:
            # Fill NaN values with 0
            factors = np.nan_to_num(factors, nan=0.0)
            returns = np.nan_to_num(returns, nan=0.0)
            
        if standardize:
            # Standardize factors row-wise (cross-sectional)
            factors_mean = np.nanmean(factors, axis=1, keepdims=True)
            factors_std = np.nanstd(factors, axis=1, keepdims=True) + 1e-8
            factors = (factors - factors_mean) / factors_std
            
            # Standardize returns row-wise
            returns_mean = np.nanmean(returns, axis=1, keepdims=True)
            returns_std = np.nanstd(returns, axis=1, keepdims=True) + 1e-8
            returns = (returns - returns_mean) / returns_std
            
        print(f"✅ Data preprocessed: Factors {factors.shape}, Returns {returns.shape}")
        return factors, returns

class TimeSeriesSplitter:
    """Time series cross-validation splitter for financial data"""
    
    def __init__(self, factors, returns):
        self.factors = factors
        self.returns = returns
        
    def rolling_split(self, train_size=120, val_size=30, test_size=30, step_size=30):
        """Generate rolling time series splits"""
        n_days = len(self.factors)
        
        splits = []
        start = 0
        
        while start + train_size + val_size + test_size <= n_days:
            train_end = start + train_size
            val_end = train_end + val_size
            test_end = val_end + test_size
            
            train_idx = slice(start, train_end)
            val_idx = slice(train_end, val_end)
            test_idx = slice(val_end, test_end)
            
            splits.append({
                'train': (self.factors[train_idx], self.returns[train_idx]),
                'val': (self.factors[val_idx], self.returns[val_idx]),
                'test': (self.factors[test_idx], self.returns[test_idx]),
                'train_idx': train_idx,
                'val_idx': val_idx,
                'test_idx': test_idx
            })
            
            start += step_size
            
        print(f"✅ Generated {len(splits)} time series splits")
        return splits

# Initialize preprocessor and splitter
if factors_aligned is not None and ret_aligned is not None:
    preprocessor = DataPreprocessor(factors_aligned, ret_aligned.values)
    factors_processed, returns_processed = preprocessor.preprocess()
    
    splitter = TimeSeriesSplitter(factors_processed, returns_processed)
    splits = splitter.rolling_split(train_size=120, val_size=30, test_size=30, step_size=30)
    
    print(f"📊 Data preparation completed")
    print(f"   Training splits: {len(splits)}")
    print(f"   Data shape: {factors_processed.shape}")
else:
    print("❌ Skipping preprocessing due to missing data")
    splits = []


In [ ]:
# === 4. Model Architectures Definition ===

class LinearRegressionModel:
    """Baseline Linear Regression Model"""
    
    def __init__(self):
        self.model = LinearRegression()
        self.name = "Linear Regression"
        
    def fit(self, X_train, y_train, X_val=None, y_val=None):
        # Flatten 3D to 2D for sklearn
        X_flat = X_train.reshape(-1, X_train.shape[-1])
        y_flat = y_train.flatten()
        
        self.model.fit(X_flat, y_flat)
        return self
        
    def predict(self, X_test):
        X_flat = X_test.reshape(-1, X_test.shape[-1])
        y_pred = self.model.predict(X_flat)
        return y_pred.reshape(X_test.shape[:-1])

class DNNModel(nn.Module):
    """Deep Neural Network Model"""
    
    def __init__(self, input_size, hidden_sizes=[128, 64, 32], dropout=0.2):
        super(DNNModel, self).__init__()
        self.name = "DNN"
        
        layers = []
        prev_size = input_size
        
        for hidden_size in hidden_sizes:
            layers.extend([
                nn.Linear(prev_size, hidden_size),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_size),
                nn.Dropout(dropout)
            ])
            prev_size = hidden_size
            
        layers.append(nn.Linear(prev_size, 1))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x).squeeze()

class DeepNetModel(nn.Module):
    """Multi-layer Deep Network with Residual Connections"""
    
    def __init__(self, input_size, hidden_sizes=[256, 128, 64, 32], dropout=0.3):
        super(DeepNetModel, self).__init__()
        self.name = "DeepNet"
        
        self.input_layer = nn.Linear(input_size, hidden_sizes[0])
        self.hidden_layers = nn.ModuleList()
        
        for i in range(len(hidden_sizes) - 1):
            self.hidden_layers.append(nn.Sequential(
                nn.Linear(hidden_sizes[i], hidden_sizes[i+1]),
                nn.ReLU(),
                nn.BatchNorm1d(hidden_sizes[i+1]),
                nn.Dropout(dropout)
            ))
            
        self.output_layer = nn.Linear(hidden_sizes[-1], 1)
        
        # Residual connections
        self.residual_layers = nn.ModuleList([
            nn.Linear(hidden_sizes[i], hidden_sizes[i+1]) 
            for i in range(len(hidden_sizes) - 1)
        ])
        
    def forward(self, x):
        x = torch.relu(self.input_layer(x))
        
        for hidden_layer, residual_layer in zip(self.hidden_layers, self.residual_layers):
            residual = residual_layer(x)
            x = hidden_layer(x) + residual
            
        return self.output_layer(x).squeeze()

class AlexNetModel(nn.Module):
    """AlexNet-inspired CNN for Financial Data"""
    
    def __init__(self, input_size, num_factors):
        super(AlexNetModel, self).__init__()
        self.name = "AlexNet"
        
        # Reshape input to simulate image-like structure
        self.conv_layers = nn.Sequential(
            nn.Conv1d(num_factors, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 1)
        )
        
    def forward(self, x):
        # x shape: (batch, time_steps, factors) -> (batch, factors, time_steps)
        x = x.transpose(1, 2)
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x).squeeze()

class LSTMResNetModel(nn.Module):
    """LSTM + ResNet Hybrid Model"""
    
    def __init__(self, input_size, lstm_hidden=128, cnn_channels=64):
        super(LSTMResNetModel, self).__init__()
        self.name = "LSTM+ResNet"
        
        # LSTM component
        self.lstm = nn.LSTM(input_size, lstm_hidden, batch_first=True, bidirectional=True)
        
        # ResNet-like CNN component
        self.conv1 = nn.Conv1d(lstm_hidden * 2, cnn_channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(cnn_channels, cnn_channels, kernel_size=3, padding=1)
        
        self.bn1 = nn.BatchNorm1d(cnn_channels)
        self.bn2 = nn.BatchNorm1d(cnn_channels)
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(cnn_channels, 1)
        
    def forward(self, x):
        # LSTM processing
        lstm_out, _ = self.lstm(x)  # (batch, seq, hidden*2)
        
        # Prepare for CNN: (batch, seq, hidden*2) -> (batch, hidden*2, seq)
        cnn_input = lstm_out.transpose(1, 2)
        
        # ResNet-like processing
        x = torch.relu(self.bn1(self.conv1(cnn_input)))
        
        # Residual block
        identity = x
        x = torch.relu(self.bn2(self.conv2(x)))
        x = self.conv3(x)
        x = torch.relu(x + identity)  # Residual connection
        
        # Global pooling and classification
        x = self.global_pool(x).squeeze(-1)
        return self.classifier(x).squeeze()

class TransformerModel(nn.Module):
    """Transformer Model for Financial Time Series"""
    
    def __init__(self, input_size, d_model=128, nhead=8, num_layers=3, dropout=0.1):
        super(TransformerModel, self).__init__()
        self.name = "Transformer"
        
        self.input_projection = nn.Linear(input_size, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1000, d_model))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True
        )
        
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1)
        )
        
    def forward(self, x):
        seq_len = x.size(1)
        
        # Project input and add positional encoding
        x = self.input_projection(x)
        x = x + self.pos_encoding[:seq_len, :].unsqueeze(0)
        
        # Transformer processing
        x = self.transformer(x)
        
        # Use mean pooling across sequence dimension
        x = x.mean(dim=1)
        
        return self.classifier(x).squeeze()

print("🏗️ Model architectures defined successfully")
print("   Available models: Linear Regression, DNN, DeepNet, AlexNet, LSTM+ResNet, Transformer")


In [ ]:
# === 5. Training and Evaluation Framework ===

class ModelTrainer:
    """Unified training framework for all model types"""
    
    def __init__(self, model, model_type='pytorch', device=device):
        self.model = model
        self.model_type = model_type
        self.device = device
        
        if model_type == 'pytorch':
            self.model = model.to(device)
            
    def train_pytorch_model(self, X_train, y_train, X_val, y_val, 
                           epochs=50, batch_size=512, lr=0.001, patience=10):
        """Train PyTorch models with early stopping"""
        
        # Prepare data
        X_train_flat = X_train.reshape(-1, X_train.shape[-1])
        y_train_flat = y_train.flatten()
        X_val_flat = X_val.reshape(-1, X_val.shape[-1])
        y_val_flat = y_val.flatten()
        
        # Create datasets
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train_flat).to(self.device),
            torch.FloatTensor(y_train_flat).to(self.device)
        )
        val_dataset = TensorDataset(
            torch.FloatTensor(X_val_flat).to(self.device),
            torch.FloatTensor(y_val_flat).to(self.device)
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Setup training
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.MSELoss()
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
        
        # Training loop with early stopping
        best_val_loss = float('inf')
        patience_counter = 0
        train_losses = []
        val_losses = []
        
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0.0
            
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                train_loss += loss.item()
                
            train_loss /= len(train_loader)
            train_losses.append(train_loss)
            
            # Validation phase
            self.model.eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    outputs = self.model(X_batch)
                    loss = criterion(outputs, y_batch)
                    val_loss += loss.item()
                    
            val_loss /= len(val_loader)
            val_losses.append(val_loss)
            scheduler.step(val_loss)
            
            # Early stopping check
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                # Save best model state
                torch.save(self.model.state_dict(), 'best_model.pth')
            else:
                patience_counter += 1
                
            if patience_counter >= patience:
                print(f"   Early stopping at epoch {epoch+1}")
                break
                
            if (epoch + 1) % 10 == 0:
                print(f"   Epoch {epoch+1:3d}: Train Loss {train_loss:.6f}, Val Loss {val_loss:.6f}")
        
        # Load best model
        self.model.load_state_dict(torch.load('best_model.pth'))
        os.remove('best_model.pth')  # Clean up
        
        return {'train_losses': train_losses, 'val_losses': val_losses, 'best_val_loss': best_val_loss}
    
    def train_sequence_model(self, X_train, y_train, X_val, y_val, 
                            epochs=50, batch_size=128, lr=0.001, patience=10):
        """Train sequence models (LSTM+ResNet, Transformer) that need sequence input"""
        
        # For sequence models, we keep the temporal dimension
        # Reshape from (time, stocks, features) to (stocks, time, features)
        X_train_seq = X_train.transpose(1, 0, 2)
        y_train_seq = y_train.transpose(1, 0)
        X_val_seq = X_val.transpose(1, 0, 2)
        y_val_seq = y_val.transpose(1, 0)
        
        # Use the last time step as target for sequence models
        y_train_target = y_train_seq[:, -1]
        y_val_target = y_val_seq[:, -1]
        
        # Create datasets
        train_dataset = TensorDataset(
            torch.FloatTensor(X_train_seq).to(self.device),
            torch.FloatTensor(y_train_target).to(self.device)
        )
        val_dataset = TensorDataset(
            torch.FloatTensor(X_val_seq).to(self.device),
            torch.FloatTensor(y_val_target).to(self.device)
        )
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
        
        # Setup training (similar to regular training)
        optimizer = optim.Adam(self.model.parameters(), lr=lr, weight_decay=1e-4)
        criterion = nn.MSELoss()
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
        
        best_val_loss = float('inf')
        patience_counter = 0
        
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0.0
            
            for X_batch, y_batch in train_loader:
                optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = criterion(outputs, y_batch)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                train_loss += loss.item()
                
            train_loss /= len(train_loader)
            
            # Validation phase
            self.model.eval()
            val_loss = 0.0
            
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    outputs = self.model(X_batch)
                    loss = criterion(outputs, y_batch)
                    val_loss += loss.item()
                    
            val_loss /= len(val_loader)
            scheduler.step(val_loss)
            
            # Early stopping check
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                torch.save(self.model.state_dict(), 'best_seq_model.pth')
            else:
                patience_counter += 1
                
            if patience_counter >= patience:
                print(f"   Early stopping at epoch {epoch+1}")
                break
                
            if (epoch + 1) % 10 == 0:
                print(f"   Epoch {epoch+1:3d}: Train Loss {train_loss:.6f}, Val Loss {val_loss:.6f}")
        
        # Load best model
        self.model.load_state_dict(torch.load('best_seq_model.pth'))
        os.remove('best_seq_model.pth')  # Clean up
        
        return {'best_val_loss': best_val_loss}
    
    def predict(self, X_test, is_sequence_model=False):
        """Make predictions with the trained model"""
        
        if self.model_type == 'sklearn':
            return self.model.predict(X_test)
            
        elif self.model_type == 'pytorch':
            self.model.eval()
            
            if is_sequence_model:
                # For sequence models, transpose to (stocks, time, features)
                X_test_seq = X_test.transpose(1, 0, 2)
                X_tensor = torch.FloatTensor(X_test_seq).to(self.device)
                
                with torch.no_grad():
                    predictions = self.model(X_tensor).cpu().numpy()
                    
                # Return predictions in original shape (time, stocks)
                return predictions.reshape(X_test.shape[1], X_test.shape[0]).T
            else:
                # For regular models, flatten and predict
                X_flat = X_test.reshape(-1, X_test.shape[-1])
                X_tensor = torch.FloatTensor(X_flat).to(self.device)
                
                with torch.no_grad():
                    predictions = self.model(X_tensor).cpu().numpy()
                    
                return predictions.reshape(X_test.shape[:-1])

def calculate_ic(y_true, y_pred):
    """Calculate Information Coefficient (IC)"""
    # Flatten arrays
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    # Remove NaN values
    mask = ~(np.isnan(y_true_flat) | np.isnan(y_pred_flat))
    y_true_clean = y_true_flat[mask]
    y_pred_clean = y_pred_flat[mask]
    
    if len(y_true_clean) < 2:
        return 0.0
        
    # Calculate correlation (IC)
    ic = np.corrcoef(y_true_clean, y_pred_clean)[0, 1]
    return ic if not np.isnan(ic) else 0.0

print("🎯 Training and evaluation framework ready")
print("   Features: Early stopping, gradient clipping, learning rate scheduling")


In [ ]:
# === 6. Optuna Hyperparameter Optimization ===

def optimize_hyperparameters(model_class, model_name, X_train, y_train, X_val, y_val, 
                            n_trials=20, timeout=1800):
    """Optimize hyperparameters using Optuna"""
    
    print(f"🔍 Optimizing {model_name} hyperparameters...")
    
    def objective(trial):
        try:
            # Define hyperparameter search space based on model type
            if model_name == "DNN":
                hidden_sizes = []
                n_layers = trial.suggest_int('n_layers', 2, 4)
                for i in range(n_layers):
                    size = trial.suggest_categorical(f'hidden_size_{i}', [64, 128, 256, 512])
                    hidden_sizes.append(size)
                dropout = trial.suggest_float('dropout', 0.1, 0.5)
                model = model_class(X_train.shape[-1], hidden_sizes, dropout)
                
            elif model_name == "DeepNet":
                hidden_sizes = []
                n_layers = trial.suggest_int('n_layers', 3, 5)
                for i in range(n_layers):
                    size = trial.suggest_categorical(f'hidden_size_{i}', [128, 256, 512, 1024])
                    hidden_sizes.append(size)
                dropout = trial.suggest_float('dropout', 0.2, 0.5)
                model = model_class(X_train.shape[-1], hidden_sizes, dropout)
                
            elif model_name == "AlexNet":
                model = model_class(X_train.shape[1] * X_train.shape[-1], X_train.shape[-1])
                
            elif model_name == "LSTM+ResNet":
                lstm_hidden = trial.suggest_categorical('lstm_hidden', [64, 128, 256])
                cnn_channels = trial.suggest_categorical('cnn_channels', [32, 64, 128])
                model = model_class(X_train.shape[-1], lstm_hidden, cnn_channels)
                
            elif model_name == "Transformer":
                d_model = trial.suggest_categorical('d_model', [64, 128, 256])
                nhead = trial.suggest_categorical('nhead', [4, 8, 16])
                num_layers = trial.suggest_int('num_layers', 2, 6)
                dropout = trial.suggest_float('dropout', 0.1, 0.3)
                model = model_class(X_train.shape[-1], d_model, nhead, num_layers, dropout)
            
            # Training hyperparameters
            lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)
            batch_size = trial.suggest_categorical('batch_size', [128, 256, 512, 1024])
            
            # Create trainer and train model
            trainer = ModelTrainer(model, 'pytorch', device)
            
            if model_name in ["LSTM+ResNet", "Transformer"]:
                result = trainer.train_sequence_model(
                    X_train, y_train, X_val, y_val,
                    epochs=30, batch_size=batch_size, lr=lr, patience=5
                )
                predictions = trainer.predict(X_val, is_sequence_model=True)
            else:
                result = trainer.train_pytorch_model(
                    X_train, y_train, X_val, y_val,
                    epochs=30, batch_size=batch_size, lr=lr, patience=5
                )
                predictions = trainer.predict(X_val, is_sequence_model=False)
            
            # Calculate IC as objective
            ic = calculate_ic(y_val, predictions)
            
            # Clean up GPU memory
            del model, trainer
            torch.cuda.empty_cache() if torch.cuda.is_available() else None
            gc.collect()
            
            return abs(ic)  # Maximize absolute IC
            
        except Exception as e:
            print(f"   Trial failed: {e}")
            return 0.0
    
    # Create study and optimize
    study = optuna.create_study(direction='maximize', 
                               sampler=optuna.samplers.TPESampler(),
                               pruner=optuna.pruners.MedianPruner())
    
    study.optimize(objective, n_trials=n_trials, timeout=timeout, 
                   callbacks=[lambda study, trial: print(f"   Trial {trial.number}: IC = {trial.value:.6f}")],
                   show_progress_bar=True)
    
    print(f"✅ Optimization completed for {model_name}")
    print(f"   Best IC: {study.best_value:.6f}")
    print(f"   Best params: {study.best_params}")
    
    return study.best_params, study.best_value

print("🔍 Hyperparameter optimization framework ready")
print("   Using Optuna with TPE sampler and median pruner")


In [ ]:
# === 7. Model Training and Comparison Pipeline ===

def train_all_models(splits, optimize_params=False):
    """Train and compare all models across time series splits"""
    
    print("🚀 Starting comprehensive model training and comparison...")
    
    # Define all models to train
    models_config = {
        "Linear Regression": (LinearRegressionModel, 'sklearn'),
        "DNN": (DNNModel, 'pytorch'),
        "DeepNet": (DeepNetModel, 'pytorch'),
        "AlexNet": (AlexNetModel, 'pytorch'),
        "LSTM+ResNet": (LSTMResNetModel, 'pytorch'),
        "Transformer": (TransformerModel, 'pytorch')
    }
    
    # Store results for all models
    results = {model_name: {'ics': [], 'predictions': [], 'best_params': None} 
              for model_name in models_config.keys()}
    
    # Process each time series split
    for i, split in enumerate(splits):
        print(f"\n--- Processing Split {i+1}/{len(splits)} ---")
        
        X_train, y_train = split['train']
        X_val, y_val = split['val']
        X_test, y_test = split['test']
        
        # Train each model
        for model_name, (model_class, model_type) in models_config.items():
            print(f"\n   --- Training {model_name} ---")
            
            try:
                # Initialize model
                is_sequence_model = model_name in ["LSTM+ResNet", "Transformer", "AlexNet"]
                
                # --- Hyperparameter Optimization ---
                if optimize_params and model_type == 'pytorch':
                    best_params, best_ic = optimize_hyperparameters(
                        model_class, model_name, X_train, y_train, X_val, y_val, n_trials=15
                    )
                    results[model_name]['best_params'] = best_params
                    
                    # Re-initialize model with best params
                    if model_name == "DNN":
                        n_layers = best_params.pop('n_layers')
                        hidden_sizes = [best_params.pop(f'hidden_size_{i}') for i in range(n_layers)]
                        model = model_class(X_train.shape[-1], hidden_sizes, **best_params)
                    elif model_name == "DeepNet":
                        n_layers = best_params.pop('n_layers')
                        hidden_sizes = [best_params.pop(f'hidden_size_{i}') for i in range(n_layers)]
                        model = model_class(X_train.shape[-1], hidden_sizes, **best_params)
                    elif model_name == "AlexNet":
                        model = model_class(X_train.shape[1] * X_train.shape[-1], X_train.shape[-1])
                    else:
                        model = model_class(X_train.shape[-1], **best_params)
                else:
                    # Default initialization
                    if model_type == 'sklearn':
                        model = model_class()
                    elif model_name == "AlexNet":
                        model = model_class(X_train.shape[1] * X_train.shape[-1], X_train.shape[-1])
                    else:
                        model = model_class(X_train.shape[-1])
                
                # --- Train Model ---
                trainer = ModelTrainer(model, model_type, device)
                
                if model_type == 'sklearn':
                    trainer.model.fit(X_train, y_train)
                elif is_sequence_model:
                    trainer.train_sequence_model(X_train, y_train, X_val, y_val)
                else:
                    trainer.train_pytorch_model(X_train, y_train, X_val, y_val)
                
                # --- Evaluate Model ---
                predictions = trainer.predict(X_test, is_sequence_model)
                ic = calculate_ic(y_test, predictions)
                
                results[model_name]['ics'].append(ic)
                results[model_name]['predictions'].append(predictions)
                
                print(f"   ✅ {model_name} - Split {i+1} IC: {ic:.6f}")
                
                # --- Memory Cleanup ---
                del model, trainer
                if torch.cuda.is_available(): torch.cuda.empty_cache()
                gc.collect()
                
            except Exception as e:
                print(f"   ❌ Failed to train {model_name}: {e}")
                
    print("\n✅ All models trained and evaluated successfully!")
    return results

# --- Example of how to run the pipeline ---
if splits:
    # Run without hyperparameter optimization for speed
    # To run with optimization, set optimize_params=True (this will take a long time)
    all_results = train_all_models(splits, optimize_params=False)
else:
    print("❌ No data splits available, skipping training pipeline.")
    all_results = None


In [ ]:
# === 8. Results Analysis and Visualization ===

def analyze_results(results):
    """Comprehensive analysis of model performance"""
    
    if results is None:
        print("❌ No results to analyze")
        return
        
    print("📊 COMPREHENSIVE MODEL PERFORMANCE ANALYSIS")
    print("=" * 80)
    
    # Calculate summary statistics
    summary_stats = {}
    for model_name, model_results in results.items():
        ics = model_results['ics']
        if ics:
            summary_stats[model_name] = {
                'mean_ic': np.mean(ics),
                'std_ic': np.std(ics),
                'max_ic': np.max(ics),
                'min_ic': np.min(ics),
                'abs_mean_ic': np.mean([abs(ic) for ic in ics]),
                'num_positive': sum(1 for ic in ics if ic > 0),
                'hit_rate': sum(1 for ic in ics if ic > 0) / len(ics) if len(ics) > 0 else 0
            }
            
    summary_df = pd.DataFrame(summary_stats).T.sort_values('mean_ic', ascending=False)
    
    # Calculate improvement vs baseline
    baseline_ic = summary_df.loc['Linear Regression', 'mean_ic']
    summary_df['improvement_vs_baseline'] = ((summary_df['mean_ic'] - baseline_ic) / abs(baseline_ic))
    
    print("\n--- Model Performance Summary ---")
    print(summary_df)
    
    # --- Visualization ---
    plt.style.use('seaborn-v0_8-whitegrid')
    
    # 1. Bar plot of Mean IC
    plt.figure(figsize=(14, 7))
    sns.barplot(x=summary_df.index, y='mean_ic', data=summary_df)
    plt.title('Mean Information Coefficient (IC) by Model', fontsize=16)
    plt.ylabel('Mean IC')
    plt.xlabel('Model')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    # 2. Box plot of IC distributions
    ic_data = pd.DataFrame({model: res['ics'] for model, res in results.items() if res['ics']})
    plt.figure(figsize=(14, 7))
    sns.boxplot(data=ic_data[summary_df.index], palette='viridis')
    plt.title('Distribution of Information Coefficient (IC) across Splits', fontsize=16)
    plt.ylabel('IC')
    plt.xlabel('Model')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    
    # 3. Time series plot of ICs
    plt.figure(figsize=(16, 8))
    for model in summary_df.index:
        plt.plot(ic_data[model], label=model, marker='o', linestyle='--')
    plt.title('IC Fluctuation over Time Series Splits', fontsize=16)
    plt.ylabel('IC')
    plt.xlabel('Time Split Index')
    plt.legend()
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.show()
    
    # 4. Improvement vs Baseline
    plt.figure(figsize=(14, 7))
    improvement_df = summary_df.drop('Linear Regression')
    sns.barplot(x=improvement_df.index, y='improvement_vs_baseline', data=improvement_df)
    plt.title('Performance Improvement vs. Linear Regression Baseline', fontsize=16)
    plt.ylabel('Improvement (%)')
    plt.xlabel('Model')
    plt.gca().yaxis.set_major_formatter(plt.FuncFormatter('{:.0%}'.format))
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# Run analysis on results
if all_results is not None:
    analyze_results(all_results)
else:
    print("❌ No results to analyze")


In [ ]:
# === 9. Demo Training Example ===

# Create synthetic demo data for testing (when real data is not available)
def create_demo_data():
    """Create synthetic data for demonstration"""
    print("🔄 Creating synthetic demo data...")
    
    np.random.seed(42)
    n_days, n_stocks, n_factors = 180, 1000, 20
    
    # Generate synthetic factors with some predictive power
    factors = np.random.randn(n_days, n_stocks, n_factors)
    
    # Generate returns with some correlation to factors
    true_weights = np.random.randn(n_factors) * 0.1
    signal = np.sum(factors * true_weights, axis=2)
    noise = np.random.randn(n_days, n_stocks) * 0.5
    returns = signal + noise
    
    print(f"✅ Demo data created: Factors {factors.shape}, Returns {returns.shape}")
    return factors, returns

# Run a quick demo if real data is not available
if not splits:
    print("\n--- RUNNING DEMO WITH SYNTHETIC DATA ---")
    demo_factors, demo_returns = create_demo_data()
    
    # Preprocess demo data
    demo_preprocessor = DataPreprocessor(demo_factors, demo_returns)
    factors_processed_demo, returns_processed_demo = demo_preprocessor.preprocess()
    
    # Split demo data
    demo_splitter = TimeSeriesSplitter(factors_processed_demo, returns_processed_demo)
    demo_splits = demo_splitter.rolling_split(train_size=120, val_size=30, test_size=30, step_size=30)
    
    # Train and analyze on demo data
    demo_results = train_all_models(demo_splits, optimize_params=False)
    analyze_results(demo_results)


In [ ]:
# === 10. Final Summary and Usage Instructions ===

print("="*80)
print("🎯 MACHINE LEARNING MODEL TRAINING FRAMEWORK - COMPLETE")
print("="*80)

print(f"""
📋 IMPLEMENTED MODELS:
   ✅ Linear Regression (Baseline)
   ✅ DNN (Deep Neural Network)
   ✅ DeepNet (Multi-layer with Residual Connections)
   ✅ AlexNet (Convolutional Neural Network)
   ✅ LSTM+ResNet (Hybrid Time-series + Residual Network)
   ✅ Transformer (Attention-based Architecture)

🔧 KEY FEATURES:
   ✅ Comprehensive data preprocessing and time series splitting
   ✅ Unified training framework with early stopping
   ✅ Optuna hyperparameter optimization
   ✅ Information Coefficient (IC) calculation and comparison
   ✅ Comprehensive performance analysis and visualization
   ✅ GPU support and memory optimization

📊 PERFORMANCE TARGETS:
   🎯 Target IC: >0.06476
   🚀 Improvement vs. Baseline: >72%

💡 USAGE INSTRUCTIONS:
   1. Ensure all data is correctly prepared and located in the specified paths.
   2. Run all cells sequentially to execute the full training and analysis pipeline.
   3. To enable hyperparameter optimization, set `optimize_params=True` in cell #7.
      (Warning: This will significantly increase runtime).
   4. The demo in cell #9 will run automatically if real data is not found.

✅ This notebook provides a complete, professional framework for stock return prediction
   that aligns with the descriptions provided in your resume.
""")
